# EDA: Score de Riesgo de Liquidez
Calcula un score numérico para alertar si el usuario no podrá cubrir sus compromisos a fin de mes.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Cargar proyección previa
output_dir = Path("../../outputs")
df_score = pd.read_csv(output_dir / "proyeccion_gastos_fin_mes.csv")


## 1. Calcular Score de Riesgo y Categorías

In [2]:
# Score actual
df_score["score_riesgo"] = (df_score["gasto_estimado_fin_mes"] + df_score["carga_fija_total"]) / df_score["ingreso_mensual_mxn"]

# Categorías
df_score["zona_riesgo"] = pd.cut(
    df_score["score_riesgo"], 
    bins=[-np.inf, 0.8, 1.0, np.inf], 
    labels=["Saludable", "Alto", "Crítico"]
)


## 2. Tendencia (Mes Anterior vs Mes Actual)

In [3]:
# Score del mes anterior (usando gasto real)
df_score["score_anterior"] = (df_score["gasto_real_mes_anterior"] + df_score["carga_fija_total"]) / df_score["ingreso_mensual_mxn"]

# Variación
df_score["delta_score_mensual"] = df_score["score_riesgo"] - df_score["score_anterior"]
df_score["tendencia_riesgo"] = np.where(df_score["delta_score_mensual"] > 0, "Empeorando", 
                                        np.where(df_score["delta_score_mensual"] < 0, "Mejorando", "Estable"))


## 3. Validación y Distribución

In [4]:
dist_zonas = df_score["zona_riesgo"].value_counts(normalize=True) * 100
print("=== Distribución de Usuarios por Zona ===")
for zona, pct in dist_zonas.items():
    count = (df_score["zona_riesgo"] == zona).sum()
    print(f"{zona}: {pct:.2f}% ({count} usuarios)")

print("\n=== Estadísticas de la Tendencia ===")
dist_tendencia = df_score["tendencia_riesgo"].value_counts(normalize=True) * 100
for tend, pct in dist_tendencia.items():
    count = (df_score["tendencia_riesgo"] == tend).sum()
    print(f"{tend}: {pct:.2f}% ({count} usuarios)")


=== Distribución de Usuarios por Zona ===
Saludable: 99.26% (14914 usuarios)
Alto: 0.47% (71 usuarios)
Crítico: 0.27% (40 usuarios)

=== Estadísticas de la Tendencia ===
Mejorando: 69.61% (10459 usuarios)
Estable: 25.84% (3882 usuarios)
Empeorando: 4.55% (684 usuarios)


## 4. Exportar a CSV

In [5]:
output_dir.mkdir(exist_ok=True)
out_path = output_dir / "score_riesgo_usuarios.csv"
df_score.to_csv(out_path, index=False)
print("\nArchivo CSV generado exitosamente en:", out_path.resolve())



Archivo CSV generado exitosamente en: D:\Datamoles\outputs\score_riesgo_usuarios.csv
